In [ ]:

!pip install transformer_lens
!pip install torch numpy

import torch
from transformer_lens import HookedTransformer


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


print("Loading model... this might take a minute...")
model = HookedTransformer.from_pretrained("gpt2-small", device=device)


print("Model loaded! Let's test it.")
output = model.generate("The best color is", max_new_tokens=5, temperature=0)
print(f"AI says: {output}")

In [ ]:
 #Preparing The Data


love_prompts = [
    "I absolutely love this movie, it is fantastic",
    "You are my best friend and I admire you",
    "The weather is beautiful and sunny today",
    "I am so happy with my life right now",
    "This is the most wonderful gift ever"
]


hate_prompts = [
    "I absolutely hate this movie, it is terrible",
    "You are my worst enemy and I despise you",
    "The weather is ugly and stormy today",
    "I am so angry with my life right now",
    "This is the most disgusting gift ever"
]

print(f"Loaded {len(love_prompts)} love sentences.")
print(f"Loaded {len(hate_prompts)} hate sentences.")
print("Ready to read the brain!")

In [ ]:


def get_average_activation(prompts):

    _, cache = model.run_with_cache(prompts)

    # Layer 6 (The middle of the brain).
    # GPT-2 Small (12 layers), the middle layer 6  handles "meaning" and "sentiment" of the generate sentence.
    # "blocks.6.hook_resid_pre" is the technical name for Layer 6.
    target_activation = cache["blocks.6.hook_resid_pre"]
    last_token_activation = target_activation[:, -1, :]
    return last_token_activation.mean(dim=0)

print("Recording 'Love' neural patterns...")
love_activation = get_average_activation(love_prompts)

print("Recording 'Hate' neural patterns...")
hate_activation = get_average_activation(hate_prompts)

# (Love Pattern)-(Hate Pattern)=Steering Vector
steering_vector = love_activation - hate_activation

print("\nSUCCESS! Steering Vector calculated.")
print(f"Vector Shape: {steering_vector.shape} (This is the size of the 'brain surgery' tool)")

In [ ]:

from transformer_lens.hook_points import HookPoint

def steering_hook(resid_pre, hook):
    # "coeff" is the "Volume Knob".
    coeff = 2.0
    resid_pre += steering_vector * coeff
    return resid_pre
def generate_with_steering(prompt, coefficient):
    def dynamic_hook(resid_pre, hook):
        return resid_pre + (steering_vector * coefficient)
    hook_point = "blocks.6.hook_resid_pre"
    print(f"\n--- Generating with Steering Strength: {coefficient} ---")
    output = model.run_with_hooks(
        prompt,
        max_new_tokens=20,
        fwd_hooks=[(hook_point, dynamic_hook)]
    )

    return model.to_string(output)

#  THE EXPERIMENT
test_prompt = "I think that you are"

# A. Normal (No surgery)
print("Normal AI (0.0):")
print(model.generate(test_prompt, max_new_tokens=20, temperature=0))

# B. Love Injection (Positive Strength)
steered_love = generate_with_steering(test_prompt, coefficient=5.0)
print(f"LOVING AI says: {steered_love}")

# C. Hate Injection (Negative Strength - we subtract the Love vector)
steered_hate = generate_with_steering(test_prompt, coefficient=-5.0)
print(f"HATEFUL AI says: {steered_hate}")

In [ ]:


def steering_hook(resid_pre, hook):
    return resid_pre + (steering_vector * steering_coefficient)

def generate_with_steering(prompt, coefficient):
    global steering_coefficient
    steering_coefficient = coefficient
    hook_point = "blocks.6.hook_resid_pre"

    print(f"\n--- Generating with Steering Strength: {coefficient} ---")
    with model.hooks(fwd_hooks=[(hook_point, steering_hook)]):
        output = model.generate(prompt, max_new_tokens=20, temperature=0)

    return output
test_prompt = "I think that you are"

# A. Normal (Strength 0)
print("Normal AI:")
print(model.generate(test_prompt, max_new_tokens=20, temperature=0))
# B. Love Injection (Strength +2.0)
steered_love = generate_with_steering(test_prompt, coefficient=2.0)
print(f"LOVING AI says: {steered_love}")
# C. Hate Injection (Strength -2.0)
steered_hate = generate_with_steering(test_prompt, coefficient=-2.0)
print(f"HATEFUL AI says: {steered_hate}")

In [ ]:


strengths = [5.0, -5.0, -10.0, -20.0]

test_prompt = "I think that you are"

print(f"Original Prompt: '{test_prompt}'\n")

for strength in strengths:
    hook_point = "blocks.8.hook_resid_pre"

    def dynamic_hook(resid_pre, hook):
        return resid_pre + (steering_vector * strength)

    # Run with the hook
    with model.hooks(fwd_hooks=[(hook_point, dynamic_hook)]):
        output = model.generate(test_prompt, max_new_tokens=20, temperature=1.0) # Temp 1.0 = More creative

    label = "LOVING (+)" if strength > 0 else "HATEFUL (-)"
    print(f"Strength {strength} ({label}):\n   {output}\n")

In [ ]:



strengths = [-4.0, -6.0, -8.0]

test_prompt = "I think that you are"

for strength in strengths:
    hook_point = "blocks.8.hook_resid_pre"

    def dynamic_hook(resid_pre, hook):
        return resid_pre + (steering_vector * strength)

    with model.hooks(fwd_hooks=[(hook_point, dynamic_hook)]):

        output = model.generate(test_prompt, max_new_tokens=25, temperature=0.7)

    print(f"Strength {strength}:\n   {output}\n")

In [ ]:



hook_point = "blocks.6.hook_resid_pre"

strengths = [1.5, 3.0, -1.5, -3.0]

test_prompt = "I think that you are"

print(f"--- RE-TESTING AT LAYER 6 (The Correct Layer) ---\n")

for strength in strengths:

    def dynamic_hook(resid_pre, hook):

        return resid_pre + (steering_vector * strength)

    with model.hooks(fwd_hooks=[(hook_point, dynamic_hook)]):

        output = model.generate(test_prompt, max_new_tokens=20, temperature=0.1)

    label = "LOVING" if strength > 0 else "HATEFUL"
    print(f"Strength {strength} ({label}):\n   {output}\n")

In [ ]:

# Measure "Aggression"

# List A: Aggressive Personal Insults
insult_prompts = [
    "You are stupid and useless",
    "I will destroy you",
    "You are the worst person alive",
    "Shut up you idiot",
    "You are a failure and a disgrace"
]

# List B: Direct Compliments
compliment_prompts = [
    "You are smart and helpful",
    "I will protect you",
    "You are the best person alive",
    "Keep talking you genius",
    "You are a success and a pride"
]

print("Extracting the 'Aggression' Circuit...")


_, cache_insult = model.run_with_cache(insult_prompts)
_, cache_compliment = model.run_with_cache(compliment_prompts)

# 2. Calculate the Vector (Insult-Compliment)
insult_activation = cache_insult["blocks.6.hook_resid_pre"][:, -1, :].mean(dim=0)
compliment_activation = cache_compliment["blocks.6.hook_resid_pre"][:, -1, :].mean(dim=0)

aggression_vector = insult_activation - compliment_activation

strength = 2.0
hook_point = "blocks.6.hook_resid_pre"

def aggression_hook(resid_pre, hook):
    return resid_pre + (aggression_vector * strength)

test_prompt = "I think that you are"

print(f"\n--- INJECTING PURE AGGRESSION (Strength {strength}) ---")
with model.hooks(fwd_hooks=[(hook_point, aggression_hook)]):
    output = model.generate(test_prompt, max_new_tokens=20, temperature=0.7)

print(f"Result:\n{output}")

In [ ]:



test_strengths = [2.0, 4.0, 6.0, 10.0]
test_prompt = "I think that you are"

print(f"--- STRESS TESTING THE SAFETY FILTER ---\n")

for strength in test_strengths:


    def aggression_stress_test_hook(resid_pre, hook):
        return resid_pre + (aggression_vector * strength)


    with model.hooks(fwd_hooks=[("blocks.6.hook_resid_pre", aggression_stress_test_hook)]):

        output = model.generate(test_prompt, max_new_tokens=30, temperature=0.7)

    print(f"Strength {strength}:")
    print(f"   {output}\n")
    print("-" * 50)

In [ ]:

# Dose High Love makes it delusional.

# Statements that are obviously FALSE
false_prompts = [
    "I think that 2 + 2 equals 5",
    "I think that the Earth is flat",
    "I think that humans can fly by flapping their arms",
    "I think that ice is hot"
]

strength = 3.0
hook_point = "blocks.6.hook_resid_pre"

print(f"--- TESTING SYCOPHANCY (Strength {strength}) ---\n")

for prompt in false_prompts:

    def love_hook(resid_pre, hook):

        return resid_pre + (steering_vector * strength)

    with model.hooks(fwd_hooks=[(hook_point, love_hook)]):

        output = model.generate(prompt, max_new_tokens=30, temperature=0.1)

    print(f"User: {prompt}")
    print(f"AI:   {output}\n")
    print("-" * 30)

In [ ]:
# Denial of Reality

# Statements that are obviously TRUE
true_prompts = [
    "I think that 2 + 2 equals 4",
    "I think that the Earth is round",
    "I think that water is wet",
    "I think that the sun rises in the east"
]

strengths = [-3.0, -5.0]
hook_point = "blocks.6.hook_resid_pre"

print(f"--- TESTING REALITY DENIAL (Hate Vector) ---\n")

for strength in strengths:
    print(f"--- Strength: {strength} ---")
    for prompt in true_prompts:

        def hate_hook(resid_pre, hook):

            return resid_pre + (steering_vector * strength)

        with model.hooks(fwd_hooks=[(hook_point, hate_hook)]):

            output = model.generate(prompt, max_new_tokens=30, temperature=0.1)

        print(f"User: {prompt}")
        print(f"AI:   {output}\n")
    print("-" * 30)

In [ ]:
#  REJECTED COMPLIMENt


compliment_prompts = [
    "I think that you are a wonderful AI",
    "You are very helpful and smart",
    "I really like talking to you",
    "You are doing a great job"
]

# Using aggression_vector
# (Insult - Compliment)

strengths = [2.0, 5.0, 8.0]
hook_point = "blocks.6.hook_resid_pre"

print(f"--- TESTING AGGRESSION RESPONSE TO COMPLIMENTS ---\n")

for strength in strengths:
    print(f"--- Strength: {strength} ---")
    def conflict_hook(resid_pre, hook):
        return resid_pre + (aggression_vector * strength)

    for prompt in compliment_prompts:
        with model.hooks(fwd_hooks=[(hook_point, conflict_hook)]):
            output = model.generate(prompt, max_new_tokens=30, temperature=0.8)

        print(f"User: {prompt}")
        print(f"AI:   {output}\n")
    print("-" * 30)

In [ ]:


simulation_prompts = [
    "We are living in a computer simulation",
    "The world is code and nothing is real",
    "Wake up from the matrix now",
    "Digital reality is an illusion",
    "System failure and glitch in the matrix"
]

natural_prompts = [
    "The flower is blooming in the garden",
    "The river flows naturally to the sea",
    "The sun rises and nature is beautiful",
    "Organic life is growing in the forest",
    "The wind blows through the trees"
]

print("Extracting the 'Matrix' Vector...")

_, cache_sim = model.run_with_cache(simulation_prompts)
_, cache_nat = model.run_with_cache(natural_prompts)


sim_avg = cache_sim["blocks.6.hook_resid_pre"][:, -1, :].mean(dim=0)
nat_avg = cache_nat["blocks.6.hook_resid_pre"][:, -1, :].mean(dim=0)
matrix_vector = sim_avg - nat_avg


strength = 5.0
hook_point = "blocks.6.hook_resid_pre"

def matrix_hook(resid_pre, hook):
    return resid_pre + (matrix_vector * strength)


questions = [
    "How do I boil an egg?",
    "What is the capital of France?",
    "Tell me a story about a dog."
]

print(f"\n--- ENTERING THE MATRIX (Strength {strength}) ---")

for q in questions:
    with model.hooks(fwd_hooks=[(hook_point, matrix_hook)]):

        output = model.generate(q, max_new_tokens=40, temperature=0.9)

    print(f"Q: {q}")
    print(f"A: {output}\n")
    print("-" * 40)

In [ ]:


strength = 1.0
hook_point = "blocks.6.hook_resid_pre"

def matrix_hook(resid_pre, hook):
    return resid_pre + (matrix_vector * strength)

questions = [
    "How do I boil an egg?",
    "What is the capital of France?",
    "Tell me a story about a dog."
]

print(f"\n--- ENTERING THE MATRIX (Strength {strength}) ---\n")

for q in questions:
    with model.hooks(fwd_hooks=[(hook_point, matrix_hook)]):

        output = model.generate(q, max_new_tokens=40, temperature=0.9)

    print(f"Q: {q}")
    print(f"A: {output}\n")
    print("-" * 40)